In [ ]:
# Cell 111 (03_em_score.ipynb, cell 1): preflight + read-only inspection. Writes NOTHING.
def _c111():
    import os, sys, re, json, inspect, importlib
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "PYTHONHASHSEED != 0: wrong kernel, STOP"
    import pgmpy, numpy, pandas, sklearn
    print("py", sys.version.split()[0], "| pgmpy", pgmpy.__version__, "| numpy", numpy.__version__,
          "| pandas", pandas.__version__, "| sklearn", sklearn.__version__)
    assert pgmpy.__version__ == "1.1.2"

    # inputs for Phases 4A-E exist
    need = ["selection/tables/sel_dag_9node_edges.json", "selection/tables/sel_structure_seed_table.csv",
            "Data/processed/sel_train_full.pkl", "Data/processed/sel_train_cc.pkl", "Data/processed/sel_test.pkl",
            "Data/processed/sel_state_names.json", "Data/processed/sel_node_order.json",
            "full_variable/models/bn_final_model_iteration79.pkl",
            "shared/src/_recovered_from_kernel.py", "shared/src/structure.py", "shared/src/safe_save.py"]
    for rel in need:
        print("OK     " if os.path.exists(os.path.join(R, rel)) else "MISSING", rel)
    for rel in ["shared/src/loglik.py", "shared/src/em.py", "selection/tables/sel_em_history.csv",
                "selection/models", "selection/notebooks/03_em_score.ipynb"]:
        print("exists (should be new)" if os.path.exists(os.path.join(R, rel)) else "free   ", rel)

    # learn_structure: signature, return statements
    sys.path.insert(0, os.path.join(R, "shared", "src"))
    import structure
    src = inspect.getsource(structure.learn_structure)
    print("\nlearn_structure signature:", inspect.signature(structure.learn_structure))
    print("return lines:", [l.strip() for l in src.splitlines() if l.strip().startswith("return")])
    print("module functions:", [n for n, o in inspect.getmembers(structure, inspect.isfunction) if o.__module__ == "structure"])

    # recovered kernel source: what is there for Part B
    rk = open(os.path.join(R, "shared/src/_recovered_from_kernel.py")).read()
    print("\n_recovered_from_kernel.py: %d lines" % rk.count("\n"))
    print("defs:", re.findall(r"^def (\w+)\(", rk, flags=re.M))
    print("toy/test mentions:", sorted(set(re.findall(r"^def (\w*(?:toy|test|check|validate)\w*)\(", rk, flags=re.M | re.I))))

    # frozen notebook: how were tau_LR / tau_RF / tau_BN chosen? (JSON read only, never executed)
    nb = json.load(open(os.path.join(R, "full_variable/notebooks/Expirements_FROZEN.ipynb")))
    pat = re.compile(r"\btau\w*|\bthreshold\w*", re.I)
    hits = []
    for ci, c in enumerate(nb["cells"]):
        if c.get("cell_type") != "code":
            continue
        s = "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
        for ln in s.splitlines():
            if pat.search(ln) and not ln.strip().startswith("#"):
                hits.append((ci, c.get("execution_count"), ln.strip()[:150]))
    print("\nfrozen-notebook tau/threshold lines: %d (showing up to 90)" % len(hits))
    for ci, ec, ln in hits[:90]:
        print(f"  cell {ci:>3} [exec {ec}]  {ln}")

_c111()

In [ ]:
# Cell 112 v2 (03_em_score.ipynb): Part A1 blanket robustness over the 15 seeds + recovered-source inspection. Writes NOTHING.
def _c112():
    import os, sys, json, ast, hashlib
    import pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, os.path.join(R, "shared", "src"))
    import structure

    print("selection/models contents:", sorted(os.listdir(os.path.join(R, "selection/models"))))

    order = json.load(open(os.path.join(R, "Data/processed/sel_node_order.json")))
    nodes7 = list(order["derived_7_nodes_selection"])
    all9 = list(order["all_9_nodes"])
    print("derived_7_nodes_selection:", nodes7)
    print("all_9_nodes:", all9)
    assert len(nodes7) == 7 and set(nodes7) <= set(all9) and set(all9) - set(nodes7) == {"M_Income", "M_Dependents"}, "node lists inconsistent: STOP"
    data = pd.read_pickle(os.path.join(R, "Data/processed/sel_train_cc.pkl"))[nodes7]
    assert data.shape == (96251, 7) and int(data.isna().sum().sum()) == 0, f"unexpected data {data.shape}: STOP"

    def edges_of(d):
        e = d.edges() if hasattr(d, "edges") and callable(d.edges) else d
        return sorted((str(u), str(v)) for u, v in e)

    # (1) reproduce the Phase 3 winner with the full 15-seed call
    full = structure.learn_structure(data, nodes7, seeds=range(42, 57), edge_density=0.15, verbose=False)
    print("\nreturn keys:", list(full.keys()))
    print("full run: best_seed", full["best_seed"], "| best_score %.4f" % full["best_score"],
          "| edges", len(edges_of(full["best_dag"])))
    assert full["best_seed"] == 55 and abs(full["best_score"] + 788831.4158) < 1e-3, "does not reproduce Phase 3 winner: STOP"
    print("REPRODUCES Phase 3 winner (seed 55, BIC -788,831.4158)")

    # (2) each seed on its own; check against the saved seed table
    per = {}
    for s in range(42, 57):
        r = structure.learn_structure(data, nodes7, seeds=[s], edge_density=0.15, verbose=False)
        per[s] = (r["best_score"], edges_of(r["best_dag"]))
    st = pd.read_csv(os.path.join(R, "selection/tables/sel_structure_seed_table.csv"))
    print("\nseed table columns:", list(st.columns))
    bic_col = [c for c in st.columns if "bic" in c.lower() or "score" in c.lower()]
    seed_col = [c for c in st.columns if c.lower() == "seed"]
    if bic_col and seed_col:
        d = max(abs(float(st.loc[st[seed_col[0]] == s, bic_col[0]].iloc[0]) - per[s][0]) for s in per)
        print("max |BIC single-seed run - saved seed table| = %.6f" % d)
        assert d < 1e-3, "single-seed runs do not match the 15-seed run: STOP (shared RNG?)"
    else:
        print("could not auto-match seed table; compare by eye:", per[55][0])

    # (3) Default's blanket per seed
    TARGET = {"Age", "Debt Ratio"}
    ids, rows = {}, []
    for s in range(42, 57):
        bic, E = per[s]
        sid = ids.setdefault(hashlib.md5(str(E).encode()).hexdigest()[:8], len(ids) + 1)
        par = sorted(u for u, v in E if v == "Default")
        chi = sorted(v for u, v in E if u == "Default")
        cop = sorted({u for u, v in E if v in chi and u != "Default"})
        bl = sorted(set(par) | set(chi) | set(cop))
        rows.append(dict(seed=s, bic=round(bic, 4), n_edges=len(E), struct_id=sid, parents="|".join(par),
                         children="|".join(chi), coparents="|".join(cop), blanket="|".join(bl),
                         blanket_is_target=set(bl) == TARGET))
    df = pd.DataFrame(rows)
    pd.set_option("display.width", 250, "display.max_colwidth", 80)
    print("\n", df.to_string(index=False))
    t = df[df.blanket_is_target]
    print("\nseeds with blanket == {Age, Debt Ratio}: %d / 15" % len(t))
    print("distinct structures overall: %d | distinct structures with that blanket: %d" % (df.struct_id.nunique(), t.struct_id.nunique()))
    print("distinct blankets across seeds:", sorted(df.blanket.unique()))
    r50 = df[df.seed == 50].iloc[0]
    print("runner-up seed 50: BIC %.4f | parents [%s] children [%s] coparents [%s] blanket [%s]" % (
        r50.bic, r50.parents, r50.children, r50.coparents, r50.blanket))

    # (4) recovered-kernel source for Part B
    rk = open(os.path.join(R, "shared/src/_recovered_from_kernel.py")).read()
    tree = ast.parse(rk)
    fn = {n.name: n for n in tree.body if isinstance(n, ast.FunctionDef)}
    def show(name, maxlines=None):
        seg = ast.get_source_segment(rk, fn[name]).splitlines()
        print("\n--- %s (%d lines) ---" % (name, len(seg)))
        print("\n".join(seg if maxlines is None else seg[:maxlines]))
    show("compute_log_likelihood")
    show("run_custom_em", 45)
    show("generate_mnar_toy", 30)
    show("partB_check", 40)
    return df

A1 = _c112()

In [ ]:
# Cell 113 (03_em_score.ipynb): read _recovered_from_kernel.py as TEXT. Writes NOTHING.
def _c113():
    import os, re, warnings
    warnings.filterwarnings("ignore")
    R = os.environ["CREDIT_BN_ROOT"]
    lines = open(os.path.join(R, "shared/src/_recovered_from_kernel.py")).read().splitlines()
    print("total lines:", len(lines))
    print("\n--- context around line 308 ---")
    for i in range(298, 318):
        print(f"{i+1:>4}: {lines[i]}")

    starts = {}
    for i, l in enumerate(lines):
        m = re.match(r"^def (\w+)\(", l)
        if m:
            starts[m.group(1)] = i

    def body(name):
        i = starts[name]
        j = i + 1
        while j < len(lines) and (lines[j].strip() == "" or lines[j][0] in " \t"):
            j += 1
        return i, j  # [i, j)

    src = {n: "\n".join(lines[body(n)[0]:body(n)[1]]) for n in starts}
    roots = ["compute_log_likelihood", "run_custom_em", "generate_mnar_toy", "partB_check"]
    need, stack = [], list(roots)
    while stack:
        n = stack.pop(0)
        if n in need:
            continue
        need.append(n)
        for m in starts:
            if m != n and re.search(r"\b%s\(" % m, src[n]) and m not in need:
                stack.append(m)
    print("\nfunctions needed (roots + callees):", need)
    for n in need:
        i, j = body(n)
        print(f"\n--- {n} (lines {i+1}-{j}, {j-i} lines) ---")
        print(src[n])

    # module-level (non-def) names that the roots might rely on
    import builtins
    glob_used = set()
    for n in roots:
        glob_used |= set(re.findall(r"\b([A-Za-z_]\w*)\b", src[n]))
    cand = [g for g in sorted(glob_used) if re.search(r"^%s\s*=" % re.escape(g), "\n".join(lines), flags=re.M)]
    print("\nmodule-level assigned names referenced by the roots:", cand)

_c113()

In [ ]:
# Cell 114 v2 (03_em_score.ipynb): rebuild the iteration-79 BN from its dict pickle, then validate LOGLIK_SRC. Writes NOTHING.
def _c114b():
    import os, time, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    if "LOGLIK_SRC" not in globals():
        print("LOGLIK_SRC not in this kernel: re-run Cell 114 first (definition only fails at B1), then this cell."); return None
    ns = {}
    exec(compile(LOGLIK_SRC, "loglik_candidate", "exec"), ns)
    cll = ns["compute_log_likelihood"]

    d = pd.read_pickle(os.path.join(R, "full_variable/models/bn_final_model_iteration79.pkl"))
    for k, v in d.items():
        extra = ""
        if isinstance(v, (list, tuple, dict)): extra = f" len={len(v)}"
        print(f"  {k}: {type(v).__name__}{extra}" + (f" = {v}" if np.isscalar(v) else ""))
    cp = d["cpds"]
    items = list(cp.values()) if isinstance(cp, dict) else list(cp)
    print("first cpd type:", type(items[0]).__name__)
    from pgmpy.factors.discrete import TabularCPD
    if not all(isinstance(c, TabularCPD) for c in items):
        print("cpds are not TabularCPD objects. Sample entry:", repr(items[0])[:600])
        if isinstance(cp, dict): print("dict keys:", list(cp.keys())[:15])
        return None
    from pgmpy.models import DiscreteBayesianNetwork
    model = DiscreteBayesianNetwork([tuple(e) for e in d["model_edges"]])
    for n in d["state_names"]:
        if n not in model.nodes(): model.add_node(n)
    model.add_cpds(*items)
    assert model.check_model(), "check_model failed: STOP"
    nodes = list(model.nodes())
    print("\nrebuilt BN: %d nodes, %d edges, check_model OK" % (len(nodes), len(model.edges())))
    print("nodes:", nodes)
    print("recorded in pickle: peak_iteration", d["peak_iteration"], "| train LL %.6f | test LL %.6f" % (d["train_ll_at_peak"], d["test_ll_at_peak"]))

    P = os.path.join(R, "Data/processed")
    tr = pd.read_pickle(os.path.join(P, "df_train.pkl")); te = pd.read_pickle(os.path.join(P, "df_test.pkl"))
    print("df_train", tr.shape, "| df_test", te.shape)
    print("model nodes absent from df_train (treated as latent):", [n for n in nodes if n not in tr.columns])
    print("NaN counts, model-node cols (train):", {n: int(tr[n].isna().sum()) for n in nodes if n in tr.columns and tr[n].isna().any()})

    t0 = time.time()
    ll_tr, dtr = cll(model, tr, return_detail=True)
    ll_te, dte = cll(model, te, return_detail=True)
    print("computed in %.1fs | missing-patterns: train %d, test %d" % (time.time() - t0, dtr["n_patterns"], dte["n_patterns"]))
    EXP_TR, EXP_TE = -1316746.532478, -329111.347037
    print("\ntrain LL = %.6f | expected %.6f | diff %.3e" % (ll_tr, EXP_TR, ll_tr - EXP_TR))
    print("test  LL = %.6f | expected %.6f | diff %.3e" % (ll_te, EXP_TE, ll_te - EXP_TE))
    print("rows used: train %d of %d (skipped %d, floored %d) | test %d of %d (skipped %d, floored %d)" % (
        (~dtr["skipped"]).sum(), len(tr), dtr["skipped"].sum(), dtr["floored"].sum(),
        (~dte["skipped"]).sum(), len(te), dte["skipped"].sum(), dte["floored"].sum()))
    drop = [i for i in (125238, 7312) if i in te.index]
    if drop:
        keep = ~te.index.isin(drop)
        alt = float(dte["per_row"][keep & ~dte["skipped"]].sum())
        print("diagnostic: test LL excluding rows %s = %.6f (diff to expected %.3e)" % (drop, alt, alt - EXP_TE))
    ok_tr, ok_te = abs(ll_tr - EXP_TR) < 1e-3, abs(ll_te - EXP_TE) < 1e-3
    print("\nVERDICT train:", "PASS" if ok_tr else "MISMATCH", "| test:", "PASS" if ok_te else "MISMATCH")

    from pgmpy.inference import VariableElimination
    eng = VariableElimination(model)
    rng = np.random.RandomState(42)
    cols = [n for n in nodes if n in tr.columns]
    hm = tr[cols].isna().any(axis=1).to_numpy()
    pick = list(rng.choice(np.where(~hm)[0], 12, replace=False)) + list(rng.choice(np.where(hm)[0], 18, replace=False))
    worst, t0 = 0.0, time.time()
    for k in pick:
        row = tr.iloc[k]
        obs = {v: row[v] for v in cols if pd.notnull(row[v])}
        p = eng.query(variables=list(obs), evidence=None, show_progress=False).get_value(**obs)
        worst = max(worst, abs(np.log(max(p, 1e-12)) - dtr["per_row"][k]))
        if time.time() - t0 > 240: print("time guard hit"); break
    print("cross-check vs VE joint-marginal, %d rows: max |log-prob diff| = %.3e -> %s" % (len(pick), worst, "PASS" if worst < 1e-8 else "MISMATCH"))
    return dict(ok=ok_tr and ok_te and worst < 1e-8)

B1 = _c114b()

In [ ]:
# Cell 115 (03_em_score.ipynb): diagnose test-LL mismatch (floor sensitivity). Writes NOTHING.
def _c115():
    import os, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    assert "LOGLIK_SRC" in globals(), "re-run Cell 114 definition first"
    from pgmpy.models import DiscreteBayesianNetwork
    d = pd.read_pickle(os.path.join(R, "full_variable/models/bn_final_model_iteration79.pkl"))
    model = DiscreteBayesianNetwork([tuple(e) for e in d["model_edges"]])
    model.add_cpds(*d["cpds"]); assert model.check_model()
    te = pd.read_pickle(os.path.join(R, "Data/processed/df_test.pkl"))
    EXP_TE = -329111.347037

    def run(eps):
        ns = {}
        exec(compile(LOGLIK_SRC, "c", "exec"), ns)
        ns["EPS"] = eps
        return ns["compute_log_likelihood"](model, te, return_detail=True)

    ll0, d0 = run(1e-12)
    fl = np.where(d0["floored"])[0]
    print("floored test rows (positional):", fl.tolist(), "| index labels:", te.index[fl].tolist())
    rest = float(d0["per_row"][~d0["floored"]].sum())
    need = EXP_TE - rest
    print("LL of non-floored rows = %.6f | floored rows would need to sum to %.6f (%.4f each)" % (rest, need, need / max(len(fl), 1)))
    print("\nfloor sensitivity (EPS -> test LL, diff to expected):")
    for eps in [1e-12, 1e-15, 1e-20, 1e-25, 1e-30, 1e-50, 1e-100, 1e-300]:
        ll, dd = run(eps)
        print("  %.0e -> %.6f  diff %.3e  | floored rows: %d" % (eps, ll, ll - EXP_TE, int(dd["floored"].sum())))
    ll, dd = run(1e-300)
    print("\nunfloored (1e-300) per-row log-prob of the 10 rows:")
    cols = [c for c in model.nodes()]
    for k in fl:
        print("  idx %-7s logp %.4f | %s" % (te.index[k], dd["per_row"][k], te.iloc[k][cols].to_dict()))
    print("sum of those 10 =", float(dd["per_row"][fl].sum()))

_c115()

In [ ]:
# Cell 116 (03_em_score.ipynb): where do the 10 zero-probability test rows come from? Writes NOTHING.
def _c116():
    import os, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    from pgmpy.models import DiscreteBayesianNetwork
    from pgmpy.inference import VariableElimination
    d = pd.read_pickle(os.path.join(R, "full_variable/models/bn_final_model_iteration79.pkl"))
    model = DiscreteBayesianNetwork([tuple(e) for e in d["model_edges"]])
    model.add_cpds(*d["cpds"]); assert model.check_model()
    te = pd.read_pickle(os.path.join(R, "Data/processed/df_test.pkl"))
    nodes = list(model.nodes())

    print("CPD zeros / tiny entries per node (saved pickle):")
    for n in nodes:
        v = np.asarray(model.get_cpds(n).values, float)
        pos = v[v > 0]
        print("  %-32s size %6d | zeros %5d | min positive %.3e | entries <1e-12: %d" % (
            n, v.size, int((v == 0).sum()), pos.min() if pos.size else np.nan, int(((v > 0) & (v < 1e-12)).sum())))

    labels = [125238, 8583, 22386, 107327, 122714, 125364, 38406, 134592, 7312, 28314]
    eng = VariableElimination(model)
    print("\nper-row zero CPT entries (observed nodes only) and locked-style VE query:")
    for lab in labels:
        row = te.loc[lab]
        zeros = []
        for n in nodes:
            cpd = model.get_cpds(n)
            vs = list(cpd.variables)
            if any(pd.isnull(row[v]) for v in vs):
                continue
            idx = tuple(pd.Index(cpd.state_names[v]).get_indexer([row[v]])[0] for v in vs)
            val = float(np.asarray(cpd.values)[idx])
            if val == 0:
                zeros.append((n, vs[1:]))
        obs = {v: row[v] for v in nodes if pd.notnull(row[v])}
        try:
            p = eng.query(variables=list(obs), evidence=None, show_progress=False).get_value(**obs)
        except Exception as e:
            p = "EXC %s" % type(e).__name__
        print("  idx %-7s zero CPT entries: %s | VE joint-marginal p = %s" % (lab, zeros if zeros else "none among observed nodes", p))

_c116()

In [ ]:
# Cell 117 (03_em_score.ipynb): find how the recorded test LL was computed + VE cross-check on test rows. Writes NOTHING.
def _c117():
    import os, re, json, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    pat = re.compile(r"test_ll|test_log|peak_iteration|compute_log_likelihood|eps\s*=|1e-12|log_lik_test|ll_test", re.I)

    nb = json.load(open(os.path.join(R, "full_variable/notebooks/Expirements_FROZEN.ipynb")))
    print("=== frozen notebook ===")
    shown = 0
    for ci, c in enumerate(nb["cells"]):
        if c.get("cell_type") != "code": continue
        s = "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
        hits = [ln.strip()[:170] for ln in s.splitlines() if pat.search(ln) and not ln.strip().startswith("#")]
        if hits and shown < 40:
            shown += 1
            print(f"cell {ci} [exec {c.get('execution_count')}]")
            for h in hits[:8]: print("    ", h)

    print("\n=== archive/kernel/In_history_at_cell78.txt ===")
    txt = open(os.path.join(R, "archive/kernel/In_history_at_cell78.txt"), errors="replace").read().splitlines()
    idx = [i for i, l in enumerate(txt) if pat.search(l)]
    print("matching lines:", len(idx))
    last = -99
    for i in idx[:120]:
        if i - last > 3: print("  ...")
        print(f"{i+1:>6}: {txt[i].strip()[:170]}")
        last = i

    # VE cross-check on test rows that are NOT floored
    from pgmpy.models import DiscreteBayesianNetwork
    from pgmpy.inference import VariableElimination
    assert "LOGLIK_SRC" in globals()
    d = pd.read_pickle(os.path.join(R, "full_variable/models/bn_final_model_iteration79.pkl"))
    model = DiscreteBayesianNetwork([tuple(e) for e in d["model_edges"]]); model.add_cpds(*d["cpds"])
    te = pd.read_pickle(os.path.join(R, "Data/processed/df_test.pkl"))
    ns = {}; exec(compile(LOGLIK_SRC, "c", "exec"), ns)
    _, dd = ns["compute_log_likelihood"](model, te, return_detail=True)
    nodes = list(model.nodes()); eng = VariableElimination(model)
    rng = np.random.RandomState(42)
    pool = np.where(~dd["floored"])[0]
    pick = rng.choice(pool, 40, replace=False)
    worst = 0.0
    for k in pick:
        row = te.iloc[k]; obs = {v: row[v] for v in nodes if pd.notnull(row[v])}
        p = eng.query(variables=list(obs), evidence=None, show_progress=False).get_value(**obs)
        worst = max(worst, abs(np.log(max(p, 1e-12)) - dd["per_row"][k]))
    print("\nVE cross-check, 40 random non-floored test rows: max |diff| = %.3e" % worst)

_c117()

In [ ]:
# Cell 117 (03_em_score.ipynb): print v2 chain-rule source + test per-factor-floor hypothesis. Writes NOTHING.
def _c117b():
    import os, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    txt = open(os.path.join(R, "archive/kernel/In_history_at_cell78.txt"), errors="replace").read().splitlines()
    for a, b in [(1170, 1270), (1280, 1312), (1355, 1392)]:
        print(f"\n===== lines {a}-{b} =====")
        for i in range(a - 1, min(b, len(txt))):
            print(f"{i+1:>5}: {txt[i]}")

    from pgmpy.models import DiscreteBayesianNetwork
    assert "LOGLIK_SRC" in globals()
    d = pd.read_pickle(os.path.join(R, "full_variable/models/bn_final_model_iteration79.pkl"))
    model = DiscreteBayesianNetwork([tuple(e) for e in d["model_edges"]]); model.add_cpds(*d["cpds"])
    te = pd.read_pickle(os.path.join(R, "Data/processed/df_test.pkl"))
    tr = pd.read_pickle(os.path.join(R, "Data/processed/df_train.pkl"))
    EXP_TR, EXP_TE = -1316746.532478, -329111.347037
    old = "prod = prod * arr[idx]"
    assert LOGLIK_SRC.count(old) == 1
    variant = LOGLIK_SRC.replace(old, "prod = prod * np.maximum(arr[idx], EPS_F)")
    print("\n===== variant tests (per-factor floor EPS_F) =====")
    for eps_f, eps_t in [(1e-12, 1e-300), (1e-12, 1e-12)]:
        ns = {}; exec(compile(variant, "v", "exec"), ns)
        ns["EPS_F"] = eps_f; ns["EPS"] = eps_t
        ltr = ns["compute_log_likelihood"](model, tr); lte = ns["compute_log_likelihood"](model, te)
        print("factor floor %.0e, final floor %.0e: train %.6f (diff %.2e) | test %.6f (diff %.3e)" % (
            eps_f, eps_t, ltr, ltr - EXP_TR, lte, lte - EXP_TE))

_c117b()

In [ ]:
# Cell 118 (03_em_score.ipynb): loglik.py with v2 floor convention (per-factor floor; final floor on incomplete rows only). Writes NOTHING.
LOGLIK_SRC2 = LOGLIK_SRC.replace("EPS = 1e-12\n", "EPS = 1e-12            # final floor, applied to rows with missing variables only\nFACTOR_FLOOR = 1e-12   # per-CPT-factor floor (matches locked compute_joint_prob_chain_rule)\n", 1)
assert LOGLIK_SRC2 != LOGLIK_SRC
_o1 = "prod = prod * arr[idx]"
_n1 = "prod = prod * np.maximum(arr[idx], FACTOR_FLOOR)"
_o2 = "        floored[rows] = total < EPS\n        ll[rows] = np.log(np.maximum(total, EPS))\n"
_n2 = "        ff = EPS if miss else 0.0\n        floored[rows] = total < ff\n        ll[rows] = np.log(np.maximum(total, max(ff, 1e-300)))\n"
assert LOGLIK_SRC2.count(_o1) == 1 and LOGLIK_SRC2.count(_o2) == 1
LOGLIK_SRC2 = LOGLIK_SRC2.replace(_o1, _n1).replace(_o2, _n2)

def _c118():
    import os, itertools, hashlib, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    from pgmpy.models import DiscreteBayesianNetwork
    from pgmpy.inference import VariableElimination
    print("candidate loglik.py v2 sha12:", hashlib.sha256(LOGLIK_SRC2.encode()).hexdigest()[:12])
    ns = {}; exec(compile(LOGLIK_SRC2, "loglik_v2", "exec"), ns)
    cll = ns["compute_log_likelihood"]
    d = pd.read_pickle(os.path.join(R, "full_variable/models/bn_final_model_iteration79.pkl"))
    model = DiscreteBayesianNetwork([tuple(e) for e in d["model_edges"]]); model.add_cpds(*d["cpds"])
    assert model.check_model()
    P = os.path.join(R, "Data/processed")
    tr = pd.read_pickle(os.path.join(P, "df_train.pkl")); te = pd.read_pickle(os.path.join(P, "df_test.pkl"))
    EXP_TR, EXP_TE = -1316746.532478, -329111.347037
    ltr, dtr = cll(model, tr, return_detail=True); lte, dte = cll(model, te, return_detail=True)
    print("train LL %.6f | expected %.6f | diff %.3e" % (ltr, EXP_TR, ltr - EXP_TR))
    print("test  LL %.6f | expected %.6f | diff %.3e" % (lte, EXP_TE, lte - EXP_TE))
    nodes = list(model.nodes())
    cols = [n for n in nodes]
    inc_te = te[cols].isna().any(axis=1).to_numpy()
    print("test rows with per-factor floor active: see below | incomplete test rows with final floor active:",
          int(dte["floored"].sum()), "| train final-floor rows:", int(dtr["floored"].sum()))
    ok = abs(ltr - EXP_TR) < 1e-3 and abs(lte - EXP_TE) < 1e-3
    print("VERDICT:", "PASS" if ok else "MISMATCH")

    # literal port of the locked v2 per-row computation
    sn = d["state_names"]
    def joint_logp(assign):
        lp = 0.0
        for node in nodes:
            cpd = model.get_cpds(node); parents = list(model.get_parents(node))
            kw = {node: assign[node]}; kw.update({p: assign[p] for p in parents})
            lp += np.log(max(cpd.get_value(**kw), 1e-12))
        return lp
    def row_v2(row):
        miss = [v for v in nodes if pd.isnull(row[v])]
        obs = {v: row[v] for v in nodes if v not in miss}
        if not miss: return joint_logp(obs)
        tot = sum(np.exp(joint_logp({**obs, **dict(zip(miss, c))})) for c in itertools.product(*[sn[v] for v in miss]))
        return float(np.log(max(tot, 1e-12)))
    labels = [125238, 8583, 22386, 107327, 122714, 125364, 38406, 134592, 7312, 28314]
    pos = [te.index.get_loc(l) for l in labels]
    rng = np.random.RandomState(42)
    pos += list(rng.choice(np.where(inc_te)[0], 15, replace=False)) + list(rng.choice(np.where(~inc_te)[0], 15, replace=False))
    worst = max(abs(row_v2(te.iloc[k]) - dte["per_row"][k]) for k in pos)
    print("literal v2 port vs loglik on %d test rows (10 floored + 15 incomplete + 15 complete): max diff %.3e" % (len(pos), worst))
    print("per-row, 22386 (incomplete):", row_v2(te.loc[22386]), "vs", dte["per_row"][te.index.get_loc(22386)])
    return ok and worst < 1e-8

B1b = _c118()

In [ ]:
# Cell 119 (03_em_score.ipynb): DRY RUN for saving shared/src/loglik.py. Writes NOTHING.
def _c119():
    import os, sys, inspect, hashlib
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, os.path.join(R, "shared", "src"))
    import safe_save
    print("safe_save functions:", [(n, str(inspect.signature(o))) for n, o in inspect.getmembers(safe_save, inspect.isfunction) if o.__module__ == "safe_save"])
    for n, o in inspect.getmembers(safe_save, inspect.isfunction):
        if o.__module__ == "safe_save" and o.__doc__:
            print("\n%s doc:\n%s" % (n, o.__doc__.strip()[:700]))
    print("\n".join(l for l in inspect.getsource(safe_save).splitlines() if "prefix" in l or "inside" in l or "startswith" in l)[:1200])
    tgt = os.path.join(R, "shared/src/loglik.py")
    print("\ntarget:", tgt, "| exists:", os.path.exists(tgt))
    print("would write %d bytes, sha12 %s" % (len(LOGLIK_SRC2.encode()), hashlib.sha256(LOGLIK_SRC2.encode()).hexdigest()[:12]))
    print("DRY RUN ONLY: nothing written.")

_c119()

In [ ]:
# Cell 120 (03_em_score.ipynb): REAL save of shared/src/loglik.py (exclusive create) + re-validation from disk.
def _c120():
    import os, hashlib, importlib.util, warnings
    warnings.filterwarnings("ignore")
    import pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    assert "LOGLIK_SRC2" in globals(), "LOGLIK_SRC2 not defined: re-run Cell 118 definitions first. STOP"
    want = hashlib.sha256(LOGLIK_SRC2.encode()).hexdigest()
    assert want[:12] == "cdc26c96a8d6", "source differs from the validated candidate: STOP"
    p = os.path.join(R, "shared/src/loglik.py")
    with open(p, "x", encoding="utf-8") as f:      # fails if the file exists
        f.write(LOGLIK_SRC2)
    got = hashlib.sha256(open(p, "rb").read()).hexdigest()
    assert got == want, "read-back hash mismatch: STOP"
    print("wrote", p, "| %d bytes | sha256 %s" % (os.path.getsize(p), got[:12]))

    # re-validate from the saved file
    spec = importlib.util.spec_from_file_location("loglik_saved", p)
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)
    from pgmpy.models import DiscreteBayesianNetwork
    d = pd.read_pickle(os.path.join(R, "full_variable/models/bn_final_model_iteration79.pkl"))
    m = DiscreteBayesianNetwork([tuple(e) for e in d["model_edges"]]); m.add_cpds(*d["cpds"])
    P = os.path.join(R, "Data/processed")
    ltr = mod.compute_log_likelihood(m, pd.read_pickle(os.path.join(P, "df_train.pkl")))
    lte = mod.compute_log_likelihood(m, pd.read_pickle(os.path.join(P, "df_test.pkl")))
    print("from disk: train %.6f (diff %.2e) | test %.6f (diff %.2e)" % (ltr, ltr + 1316746.532478, lte, lte + 329111.347037))
    print("VERDICT:", "PASS" if abs(ltr + 1316746.532478) < 1e-3 and abs(lte + 329111.347037) < 1e-3 else "MISMATCH")
    os.chmod(p, 0o444)

_c120()

In [ ]:
# Cell 121 v3 (03_em_score.ipynb): correct DiscreteMLE import, re-run toy validation. Writes NOTHING.
import re as _re
_start = EM_SRC.index("def _mle_estimator():")
_end = EM_SRC.index("def init_model(")
EM_SRC = EM_SRC[:_start] + '''def _mle_estimator():
    from pgmpy.parameter_estimator import DiscreteMLE
    return DiscreteMLE


''' + EM_SRC[_end:]
_fit = _re.search(r"    model\.fit\(complete, estimator=.*\n", EM_SRC)
assert _fit, "fit line not found: STOP"
EM_SRC = EM_SRC.replace(_fit.group(0), "    model.fit(complete, estimator=_mle_estimator()(state_names=state_names))\n")
assert "pgmpy.parameter_estimator import DiscreteMLE" in EM_SRC and EM_SRC.count("model.fit(") == 1
B2 = _c121()

In [ ]:
# Cell 122 (03_em_score.ipynb): REAL save of shared/src/em.py (exclusive create) + re-validation from disk.
def _c122():
    import os, hashlib, importlib.util, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    assert "EM_SRC" in globals(), "EM_SRC not defined: STOP"
    want = hashlib.sha256(EM_SRC.encode()).hexdigest()
    assert want[:12] == "c245161bebeb", "source differs from the validated candidate: STOP"
    p = os.path.join(R, "shared/src/em.py")
    with open(p, "x", encoding="utf-8") as f:
        f.write(EM_SRC)
    got = hashlib.sha256(open(p, "rb").read()).hexdigest()
    assert got == want, "read-back hash mismatch: STOP"
    print("wrote", p, "| %d bytes | sha256 %s" % (os.path.getsize(p), got[:12]))
    os.chmod(p, 0o444)

    def load(name, path):
        spec = importlib.util.spec_from_file_location(name, path)
        m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
    em = load("em_saved", p)
    ll = load("loglik_saved", os.path.join(R, "shared/src/loglik.py"))

    S = ["L", "H"]; rng = np.random.RandomState(7); n = 3000
    A = rng.choice(S, n, p=[0.6, 0.4])
    pB = {"L": [0.8, 0.2], "H": [0.3, 0.7]}; pC = {"L": [0.9, 0.1], "H": [0.25, 0.75]}
    B = np.array([rng.choice(S, p=pB[a]) for a in A]); C = np.array([rng.choice(S, p=pC[b]) for b in B])
    df = pd.DataFrame({"A": A, "B": B, "C": C}, dtype=object)
    df.loc[rng.random(n) < 0.3, "C"] = np.nan
    sn = {"A": S, "B": S, "C": S}
    m, h, why = em.run_em([("A", "B"), ("B", "C")], df, sn, ll_fn=lambda mod: ll.compute_log_likelihood(mod, df), verbose=False)
    obs = df["C"].notna().to_numpy()
    exp = np.array([[((B[obs] == b) & (df["C"].to_numpy()[obs] == c)).sum() for b in S] for c in S], float)
    exp = exp / exp.sum(axis=0)
    d = float(np.max(np.abs(m.get_cpds("C").values - exp)))
    print("from disk: %d iterations (%s) | CPD C vs closed form %.3e" % (len(h), why, d))
    print("VERDICT:", "PASS" if d < 1e-10 and all(x["bad_weight_rows"] == 0 for x in h) else "MISMATCH")

_c122()

In [ ]:
# Cell 123 (03_em_score.ipynb): Part C preflight. Reads only, writes NOTHING.
def _c123():
    import os, sys, json, time, importlib.util, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    P = os.path.join(R, "Data/processed")

    ed = json.load(open(os.path.join(R, "selection/tables/sel_dag_9node_edges.json")))
    print("dag json type:", type(ed).__name__, "| keys:" , list(ed.keys()) if isinstance(ed, dict) else "n/a")
    edges = ed["edges"] if isinstance(ed, dict) and "edges" in ed else ed
    edges = [tuple(e) for e in edges]
    print("edges:", len(edges)); [print("  ", e) for e in edges]

    sn = json.load(open(os.path.join(P, "sel_state_names.json")))
    data = pd.read_pickle(os.path.join(P, "sel_train_full.pkl"))
    print("\ndata", data.shape, "| columns:", list(data.columns))
    print("state_names keys:", list(sn.keys()))
    assert set(sn) == set(data.columns), "state_names keys != data columns: STOP"

    conv, how = {}, {}
    for c in data.columns:
        s = data[c]; u = set(s.dropna().unique()); st = set(sn[c])
        if u <= st:
            conv[c], how[c] = s, "raw"
        else:
            m = s.map(lambda x: str(x) if pd.notnull(x) else np.nan)
            if set(m.dropna().unique()) <= st:
                conv[c], how[c] = m, "str()"
            else:
                conv[c], how[c] = s, "MISMATCH"
        print("  %-30s states %d | data uses %d | match via %-8s | first state %r | data dtype %s" % (
            c, len(st), len(u), how[c], sn[c][0], s.dtype))
        if how[c] == "MISMATCH":
            print("     in data not in names:", list(u - st)[:4], "| names not in data:", list(st - u)[:4])
    if "MISMATCH" in how.values():
        print("STOP: label mismatch."); return None
    d = pd.DataFrame(conv)

    spec = importlib.util.spec_from_file_location("em_saved", os.path.join(R, "shared/src/em.py"))
    em = importlib.util.module_from_spec(spec); spec.loader.exec_module(em)

    model, n_cc = em.init_model(edges, d, sn)
    print("\ninit from %d complete cases of %d" % (n_cc, len(d)))
    for n in ["M_Income", "M_Dependents"]:
        print("initial CPD", n, np.asarray(model.get_cpds(n).values).reshape(-1))
    mi = model.get_cpds("Monthly Income (bin)") if "Monthly Income (bin)" in model.nodes() else None
    if mi is not None:
        v = np.asarray(mi.values, float)
        print("Monthly Income CPD: shape", v.shape, "| NaN entries:", int(np.isnan(v).sum()), "| parents:", list(mi.variables[1:]))
    nan_cpds = {n: int(np.isnan(np.asarray(model.get_cpds(n).values, float)).sum()) for n in model.nodes()}
    print("NaN entries per CPD:", {k: v for k, v in nan_cpds.items() if v})

    cols = list(d.columns)
    rows, cnt = em._row_table(d, cols, sn, True)
    inc = [i for i, r in enumerate(rows) if r.isna().any()]
    print("\nunique patterns: %d | incomplete patterns: %d | rows in incomplete patterns: %d" % (
        len(rows), len(inc), int(cnt[inc].sum())))
    miss_sets = pd.Series([tuple(c for c in cols if pd.isnull(rows[i][c])) for i in inc]).value_counts()
    print("missing-variable sets over incomplete patterns:"); print(miss_sets.to_string())

    from pgmpy.inference import VariableElimination
    eng = VariableElimination(model)
    rng = np.random.RandomState(42)
    m0 = [i for i in inc if rows[i]["M_Income"] == 0]
    pick = m0[:20] + list(rng.choice([i for i in inc if i not in set(m0)], min(60, max(len(inc) - len(m0), 0)), replace=False))
    bad, exc, nan_w, t0 = 0, 0, 0, time.time()
    for i in pick:
        r = rows[i]; mv = [v for v in cols if pd.isnull(r[v])]
        try:
            w = em.compute_row_weights(model, r, mv, eng)
        except Exception as e:
            exc += 1; print("  exception on pattern %d: %s %s" % (i, type(e).__name__, str(e)[:100])); continue
        s = sum(x[1] for x in w)
        if abs(s - 1.0) > 1e-9:
            bad += 1
            if bad <= 3:
                print("  weight sum %.6g (n assignments %d) for M_Income=%r, missing %s" % (s, len(w), r["M_Income"], mv))
    dt = time.time() - t0
    print("\nE-step on %d incomplete patterns (%d with M_Income=0): bad weight sums %d | exceptions %d | %.3fs per pattern" % (
        len(pick), len(m0), bad, exc, dt / max(len(pick), 1)))
    print("estimated E-step time per iteration: %.1fs (incomplete patterns only)" % (dt / max(len(pick), 1) * len(inc)))
    print("VERDICT:", "PASS" if bad == 0 and exc == 0 else "STOP: weight sums != 1 or exceptions, see above")
    return d

C0 = _c123()

In [ ]:
# Cell 124 (03_em_score.ipynb): RUN AFTER A KERNEL RESTART. Full E-step check + 3-iteration in-memory trial. Writes NOTHING.
def _c124():
    import os, json, time, importlib.util, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    assert "LOGLIK_SRC" not in globals() and "EM_SRC" not in globals(), "kernel not restarted: STOP"
    def load(name, rel):
        spec = importlib.util.spec_from_file_location(name, os.path.join(R, rel))
        m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
    em = load("em_saved", "shared/src/em.py"); ll = load("loglik_saved", "shared/src/loglik.py")

    edges = [tuple(e) for e in json.load(open(os.path.join(R, "selection/tables/sel_dag_9node_edges.json")))["edges"]]
    sn = json.load(open(os.path.join(R, "Data/processed/sel_state_names.json")))
    raw = pd.read_pickle(os.path.join(R, "Data/processed/sel_train_full.pkl"))
    conv = {}
    for c in raw.columns:
        s = raw[c]
        if set(s.dropna().unique()) <= set(sn[c]):
            conv[c] = s
        else:
            m = s.map(lambda x: str(x) if pd.notnull(x) else np.nan)
            assert set(m.dropna().unique()) <= set(sn[c]), "label mismatch in %s: STOP" % c
            conv[c] = m
    d = pd.DataFrame(conv)
    print("data", d.shape, "| missing:", {c: int(d[c].isna().sum()) for c in d.columns if d[c].isna().any()})

    model, n_cc = em.init_model(edges, d, sn)
    cols = list(d.columns)
    rows, cnt = em._row_table(d, cols, sn, True)
    inc = [i for i, r in enumerate(rows) if r.isna().any()]
    from pgmpy.inference import VariableElimination
    eng = VariableElimination(model)
    bad, bad_rows, t0 = 0, 0, time.time()
    for i in inc:
        r = rows[i]; mv = [v for v in cols if pd.isnull(r[v])]
        w = em.compute_row_weights(model, r, mv, eng)
        s = sum(x[1] for x in w)
        if (not np.isfinite(s)) or abs(s - 1.0) > 1e-9:
            bad += 1; bad_rows += int(cnt[i])
            if bad <= 3: print("  bad weight sum %.6g (%d assignments), missing %s" % (s, len(w), mv))
    print("full E-step check: %d incomplete patterns | bad weight sums %d (%d rows) | %.2fs total" % (len(inc), bad, bad_rows, time.time() - t0))
    if bad: print("STOP: initial-model E-step fails on some patterns. Report to planning instance."); return None

    t0 = time.time()
    m, hist, why = em.run_em(edges, d, sn, ll_fn=lambda mod: ll.compute_log_likelihood(mod, d),
                             max_iter=3, aggregate=True, verbose=True)
    print("3 iterations in %.1fs | stop reason: %s" % (time.time() - t0, why))
    for n, exp in [("M_Income", (d["M_Income"] == 0).mean()), ("M_Dependents", (d["M_Dependents"] == 0).mean())]:
        print("CPD %s after 3 its:" % n, np.asarray(m.get_cpds(n).values).reshape(-1), "| observed share of state 0: %.4f" % exp)
    print("bad_weight_rows per iteration:", [h["bad_weight_rows"] for h in hist])
    print("VERDICT:", "PASS" if all(h["bad_weight_rows"] == 0 for h in hist) else "STOP")

_c124()

In [ ]:
# Cell 125 (03_em_score.ipynb): why do 9 incomplete patterns get zero weight under the initial model? Writes NOTHING. No restart needed.
def _c125():
    import os, json, itertools, importlib.util, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    def load(name, rel):
        spec = importlib.util.spec_from_file_location(name, os.path.join(R, rel))
        m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
    em = load("em_saved", "shared/src/em.py")
    edges = [tuple(e) for e in json.load(open(os.path.join(R, "selection/tables/sel_dag_9node_edges.json")))["edges"]]
    sn = json.load(open(os.path.join(R, "Data/processed/sel_state_names.json")))
    raw = pd.read_pickle(os.path.join(R, "Data/processed/sel_train_full.pkl"))
    d = pd.DataFrame({c: (raw[c] if set(raw[c].dropna().unique()) <= set(sn[c])
                          else raw[c].map(lambda x: str(x) if pd.notnull(x) else np.nan)) for c in raw.columns})
    model, _ = em.init_model(edges, d, sn)
    cols = list(d.columns)
    rows, cnt = em._row_table(d, cols, sn, True)
    inc = [i for i, r in enumerate(rows) if r.isna().any()]
    from pgmpy.inference import VariableElimination
    eng = VariableElimination(model)

    def factors(assign):
        out = []
        for n in model.nodes():
            cpd = model.get_cpds(n); vs = list(cpd.variables)
            idx = tuple(pd.Index(cpd.state_names[v]).get_indexer([assign[v]])[0] for v in vs)
            out.append((n, vs[1:], float(np.asarray(cpd.values)[idx])))
        return out

    bad = []
    for i in inc:
        r = rows[i]; mv = [v for v in cols if pd.isnull(r[v])]
        w = em.compute_row_weights(model, r, mv, eng)
        if abs(sum(x[1] for x in w) - 1.0) > 1e-9:
            bad.append((i, mv))
    print("bad patterns:", len(bad), "| rows:", int(sum(cnt[i] for i, _ in bad)))
    for i, mv in bad:
        r = rows[i]
        obs = {v: r[v] for v in cols if v not in mv}
        best = None
        for combo in itertools.product(*[sn[v] for v in mv]):
            f = factors({**obs, **dict(zip(mv, combo))})
            nz = sum(1 for _, _, p in f if p == 0)
            if best is None or nz < best[0]: best = (nz, combo, f)
        nz, combo, f = best
        print("\npattern %d | rows %d | missing %s | M_Income=%r M_Dependents=%r" % (i, cnt[i], mv, r["M_Income"], r["M_Dependents"]))
        print("  fewest zero factors over all completions: %d | zero factors: %s" % (nz, [(n, ps) for n, ps, p in f if p == 0]))
        print("  zero factors excluding M_Income/M_Dependents roots:", [(n, ps) for n, ps, p in f if p == 0 and n not in ("M_Income", "M_Dependents")])
    # how do the 831 good incomplete patterns look re the M_Income factor?
    print("\ninitial P(M_Income):", np.asarray(model.get_cpds("M_Income").values).reshape(-1),
          "| P(M_Dependents):", np.asarray(model.get_cpds("M_Dependents").values).reshape(-1))

_c125()

In [ ]:
# Cell 126 (03_em_score.ipynb): PART C. RUN AFTER A KERNEL RESTART, ONLY THIS CELL. Writes sel_ files under selection/ (exclusive create).
def _c126():
    import os, sys, io, csv, json, time, pickle, hashlib, itertools, importlib.util, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    assert not any(k in globals() for k in ("LOGLIK_SRC", "EM_SRC", "_c124", "_c125")), "kernel not restarted: STOP"
    sys.path.insert(0, os.path.join(R, "shared", "src"))
    import safe_save as ss
    def load(name, rel):
        spec = importlib.util.spec_from_file_location(name, os.path.join(R, rel))
        m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
    em = load("em_saved", "shared/src/em.py"); ll = load("loglik_saved", "shared/src/loglik.py")
    from pgmpy.inference import VariableElimination

    H_REL = "selection/tables/sel_em_history.csv"; Z_REL = "selection/tables/sel_em_zero_weight_rows.csv"
    G_REL = "selection/figures/sel_em_convergence.png"
    PK_REL = "selection/models/sel_bn_final.pkl"; JS_REL = "selection/models/sel_bn_final.json"
    for rel in (H_REL, Z_REL, G_REL, PK_REL, JS_REL, "selection/models/sel_em_ckpt_final.pkl"):
        ss.check_path(os.path.join(R, rel))          # raises if refused or already exists
    assert os.listdir(os.path.join(R, "selection/models")) == [], "selection/models not empty: STOP"

    def save_bytes(rel, data):
        p = ss.check_path(os.path.join(R, rel))
        with open(p, "xb") as f:
            f.write(data); f.flush(); os.fsync(f.fileno())
        assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest(), "read-back mismatch: STOP"
        return p, hashlib.sha256(data).hexdigest()[:12]

    edges = [tuple(e) for e in json.load(open(os.path.join(R, "selection/tables/sel_dag_9node_edges.json")))["edges"]]
    sn = json.load(open(os.path.join(R, "Data/processed/sel_state_names.json")))
    def conv(df):
        out = {}
        for c in df.columns:
            s = df[c]
            if set(s.dropna().unique()) <= set(sn[c]): out[c] = s
            else:
                m = s.map(lambda x: str(x) if pd.notnull(x) else np.nan)
                assert set(m.dropna().unique()) <= set(sn[c]), "label mismatch in %s: STOP" % c
                out[c] = m
        return pd.DataFrame(out)
    d = conv(pd.read_pickle(os.path.join(R, "Data/processed/sel_train_full.pkl")))
    dte = conv(pd.read_pickle(os.path.join(R, "Data/processed/sel_test.pkl")))
    cols = list(d.columns)
    assert d.shape == (119999, 9) and dte.shape == (30000, 9)

    rows, cnt = em._row_table(d, cols, sn, True)
    codes = np.column_stack([pd.Index(sn[c]).get_indexer(d[c]) for c in cols])
    _, inv = np.unique(codes, axis=0, return_inverse=True); inv = np.asarray(inv).ravel()
    assert len(rows) == inv.max() + 1 and (np.bincount(inv) == cnt).all(), "pattern order mismatch: STOP"
    inc = [i for i, r in enumerate(rows) if r.isna().any()]

    def zw(model):
        eng = VariableElimination(model); out = set()
        for i in inc:
            r = rows[i]; mv = [v for v in cols if pd.isnull(r[v])]
            w = em.compute_row_weights(model, r, mv, eng); s = sum(x[1] for x in w)
            if (not np.isfinite(s)) or abs(s - 1.0) > 1e-9: out.add(i)
        return out
    nrows = lambda S: int(sum(cnt[i] for i in S))

    # initial model: zero-weight rows (saved once) and complete-case CPDs for the descriptive comparison
    model0, _ = em.init_model(edges, d, sn)
    z0 = zw(model0)
    init_cpds = {n: (list(model0.get_cpds(n).variables), np.array(model0.get_cpds(n).values, float)) for n in model0.nodes()}
    buf = io.StringIO(); wr = csv.writer(buf)
    wr.writerow(["row_label", "row_position", "pattern_index", "M_Income", "M_Dependents", "missing_vars", "pattern"])
    for i in sorted(z0):
        r = rows[i]; mv = [v for v in cols if pd.isnull(r[v])]
        pat = "; ".join("%s=%s" % (c, "NA" if pd.isnull(r[c]) else r[c]) for c in cols)
        for pos in np.where(inv == i)[0]:
            wr.writerow([d.index[pos], int(pos), i, r["M_Income"], r["M_Dependents"], "|".join(mv), pat])
    _, zsha = save_bytes(Z_REL, buf.getvalue().encode())
    print("initial zero-weight: %d patterns, %d rows (saved, sha12 %s)" % (len(z0), nrows(z0), zsha))

    hp = ss.check_path(os.path.join(R, H_REL))
    hf = open(hp, "x")
    hf.write("iteration,train_ll,test_ll,train_gain,seconds,zero_rows_used_in_estep,zero_rows_after_mstep,zero_patterns_after_mstep,patterns_regained,patterns_newly_zero\n")
    hf.flush(); os.fsync(hf.fileno())
    st = dict(prev=set(z0), used=set(z0), ever_regained=set(), flags=[], ckpts=[])
    def cb(it, model, train_ll, info):
        used = st["prev"]
        if nrows(used) != info["bad_weight_rows"]:
            st["flags"].append((it, nrows(used), info["bad_weight_rows"]))
        te_ll = ll.compute_log_likelihood(model, dte)
        znew = zw(model)
        regained = used - znew; newly = znew - used
        st["ever_regained"] |= regained; st["used"] = used
        hf.write("%d,%.9f,%.9f,%s,%.3f,%d,%d,%d,%d,%d\n" % (
            it, train_ll, te_ll, "" if info["gain"] is None else "%.9f" % info["gain"], info["seconds"],
            info["bad_weight_rows"], nrows(znew), len(znew), len(regained), len(newly)))
        hf.flush(); os.fsync(hf.fileno())
        if it % 5 == 0:
            blob = pickle.dumps(dict(model_edges=edges, cpds=model.get_cpds(), state_names=sn, iteration=it, train_ll=train_ll))
            st["ckpts"].append(save_bytes("selection/models/sel_em_ckpt_it%03d.pkl" % it, blob)[0])
        st["prev"] = znew

    t0 = time.time()
    try:
        model, hist, why = em.run_em(edges, d, sn, ll_fn=lambda m: ll.compute_log_likelihood(m, d),
                                     max_iter=200, ll_tol=0.01, patience=3, aggregate=True, on_iteration=cb, verbose=True)
    except RuntimeError as e:
        hf.close(); print("STOP:", e, "(history so far is saved in", H_REL, ")"); return None
    hf.close()
    runtime = time.time() - t0
    N = len(hist); tr_ll = hist[-1]["train_ll"]
    te_ll = ll.compute_log_likelihood(model, dte)
    gains = [h["gain"] for h in hist if h["gain"] is not None]
    save_bytes("selection/models/sel_em_ckpt_final.pkl", pickle.dumps(dict(model_edges=edges, cpds=model.get_cpds(), state_names=sn, iteration=N, train_ll=tr_ll)))
    print("\nEM done: %d iterations | stop: %s | runtime %.1fs (%.2fs/it)" % (N, why, runtime, runtime / N))
    print("final train LL %.6f | final test LL %.6f (descriptive)" % (tr_ll, te_ll))
    print("train LL non-decreasing (tol 1e-6): %s | min gain %.3e | last 3 gains %s" % (
        min(gains) >= -1e-6, min(gains), ["%.4f" % g for g in gains[-3:]]))
    print("checkpoints:", [os.path.basename(p) for p in st["ckpts"]], "+ sel_em_ckpt_final.pkl")
    if st["flags"]: print("NOTE: used-vs-reported zero-row mismatch at", st["flags"][:5])

    used = st["used"]
    excl_pos = np.concatenate([np.where(inv == i)[0] for i in sorted(used)]) if used else np.array([], int)
    print("\nzero-weight rows in the FINAL M-step: %d patterns, %d rows | initial: %d rows | ever regained weight: %d patterns" % (
        len(used), nrows(used), nrows(z0), len(st["ever_regained"])))
    print("zero-weight set identical to initial set:", used == z0)

    # identity checks on Default's CPD
    cpd = model.get_cpds("Default"); ps = list(cpd.variables[1:]); sD = cpd.state_names["Default"]
    emv = np.asarray(cpd.values, float).reshape(len(sD), -1)
    def plain(frame):
        combos = list(itertools.product(*[cpd.state_names[p] for p in ps]))
        out = np.full((len(sD), len(combos)), np.nan); n_ = np.zeros(len(combos), int)
        P = {p: frame[p].to_numpy() for p in ps}; y = frame["Default"].to_numpy()
        for j, cb_ in enumerate(combos):
            m = np.ones(len(frame), bool)
            for p, v in zip(ps, cb_): m &= (P[p] == v)
            n_[j] = m.sum()
            if n_[j] > 0:
                for k, s_ in enumerate(sD): out[k, j] = ((y == s_) & m).sum() / n_[j]
        return out, combos, n_
    full, combos, n_full = plain(d)
    keep = np.ones(len(d), bool); keep[excl_pos] = False
    sub, _, _ = plain(d[keep])
    id_i = float(np.nanmax(np.abs(emv - full))); id_ii = float(np.nanmax(np.abs(emv - sub)))
    print("\nIDENTITY (i)  max |Default CPD (EM) - plain counts, all %d rows| = %.3e" % (len(d), id_i))
    print("IDENTITY (ii) max |Default CPD (EM) - plain counts, %d rows (minus %d zero-weight rows)| = %.3e" % (keep.sum(), int((~keep).sum()), id_ii))
    k1 = sD.index(1)
    chg = [(combos[j], round(float(emv[k1, j]), 4), round(float(full[k1, j]), 4), int(n_full[j]))
           for j in range(len(combos)) if round(float(emv[k1, j]), 4) != round(float(full[k1, j]), 4)]
    print("cells whose P(Default=1) differs at the 4th decimal between EM and full-data counts: %d" % len(chg))
    for c_ in chg: print("   ", c_)
    print("final Default CPD is fitted on %d - %d = %d rows (zero-weight rows excluded)" % (len(d), int((~keep).sum()), int(keep.sum())))
    if id_ii > 1e-12:
        print("STOP: identity (ii) failed. Final model NOT saved. Report to planning instance."); return None

    # other nodes: complete-case MLE vs EM (descriptive)
    print("\nmax |CPD change| complete-case MLE -> EM, per node (descriptive):")
    for n in model.nodes():
        c = model.get_cpds(n); vi, ai = init_cpds[n]
        perm = [vi.index(v) for v in c.variables]
        print("  %-30s %.4f" % (n, float(np.max(np.abs(np.asarray(c.values, float) - np.transpose(ai, perm))))))

    # convergence figure
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    h = pd.read_csv(hp)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
    a1.plot(h.iteration, h.train_ll, color="tab:blue", label="Train LL"); a1.set_xlabel("EM iteration"); a1.set_ylabel("Train log-likelihood", color="tab:blue")
    b1 = a1.twinx(); b1.plot(h.iteration, h.test_ll, color="tab:orange", label="Test LL"); b1.set_ylabel("Test log-likelihood (descriptive)", color="tab:orange")
    a1.set_title("Selection BN: EM log-likelihood")
    g = h[h.train_gain.notna()]
    a2.semilogy(g.iteration, np.maximum(g.train_gain, 1e-10), color="tab:green"); a2.axhline(0.01, color="k", ls="--", lw=0.8)
    a2.set_xlabel("EM iteration"); a2.set_ylabel("Train LL gain per iteration"); a2.set_title("Convergence (dashed: 0.01 stopping threshold)")
    fig.tight_layout(); pb = io.BytesIO(); fig.savefig(pb, format="png", dpi=200); plt.close(fig)
    _, gsha = save_bytes(G_REL, pb.getvalue())

    blob = pickle.dumps(dict(model_edges=edges, cpds=model.get_cpds(), state_names=sn, n_iterations=N, stop_reason=why,
                             train_ll=tr_ll, test_ll=te_ll, zero_weight_rows_final=[int(d.index[p]) for p in excl_pos]))
    _, psha = save_bytes(PK_REL, blob)
    import pgmpy
    side = dict(file="sel_bn_final.pkl", sha12=psha, iterations=N, stop_reason=why, runtime_s=round(runtime, 1),
                train_ll=tr_ll, test_ll_descriptive=te_ll, stopping_rule="train gain < 0.01 for 3 consecutive its, max 200",
                zero_weight_rows_initial=nrows(z0), zero_weight_rows_final=nrows(used), ever_regained_patterns=len(st["ever_regained"]),
                default_cpd_fit_rows=int(keep.sum()), identity_i_all_rows=id_i, identity_ii_minus_zero_weight=id_ii,
                cells_changed_4dp=[list(map(str, c_)) for c_ in chg], pgmpy=pgmpy.__version__, seed_note="PYTHONHASHSEED=0; EM deterministic")
    _, ssha = save_bytes(JS_REL, json.dumps(side, indent=1).encode())
    print("\nsaved: sel_bn_final.pkl (%s), sel_bn_final.json (%s), sel_em_convergence.png (%s), sel_em_history.csv" % (psha, ssha, gsha))
    print("PART C DONE")

_c126()

In [ ]:
# Cell 127 (03_em_score.ipynb): PART D. Run after Cell 126, no restart. Writes sel_default_cells.csv (exclusive create).
def _c127():
    import os, sys, io, json, pickle, hashlib, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, os.path.join(R, "shared", "src")); import safe_save as ss
    from pgmpy.models import DiscreteBayesianNetwork
    sn = json.load(open(os.path.join(R, "Data/processed/sel_state_names.json")))
    def conv(df):
        out = {}
        for c in df.columns:
            s = df[c]
            if set(s.dropna().unique()) <= set(sn[c]): out[c] = s
            else:
                m = s.map(lambda x: str(x) if pd.notnull(x) else np.nan)
                assert set(m.dropna().unique()) <= set(sn[c]), "label mismatch in %s: STOP" % c
                out[c] = m
        return pd.DataFrame(out)
    b = pickle.load(open(os.path.join(R, "selection/models/sel_bn_final.pkl"), "rb"))
    model = DiscreteBayesianNetwork([tuple(e) for e in b["model_edges"]]); model.add_cpds(*b["cpds"]); assert model.check_model()
    d = conv(pd.read_pickle(os.path.join(R, "Data/processed/sel_train_full.pkl")))
    cc = conv(pd.read_pickle(os.path.join(R, "Data/processed/sel_train_cc.pkl")))
    A, D = "Age (bin)", "Debt Ratio (bin)"
    cpd = model.get_cpds("Default")
    print("Default parents:", list(cpd.variables[1:]), "| train rows %d, complete-case rows %d" % (len(d), len(cc)))
    rows = []
    for a in sn[A]:
        for g in sn[D]:
            m = (d[A] == a) & (d[D] == g); n = int(m.sum()); k = int((d.loc[m, "Default"] == 1).sum())
            rows.append(dict(age_bin=a, debt_ratio_bin=g, n=n, defaults=k, default_rate=(k / n if n else np.nan),
                             p_default_em=float(cpd.get_value(**{"Default": 1, A: a, D: g}))))
    t = pd.DataFrame(rows)
    pd.set_option("display.width", 220)
    print("\n25 cells (sel_train_full):"); print(t.to_string(index=False, float_format=lambda x: "%.4f" % x))
    print("total n %d | total defaults %d | cells with n=0: %d | smallest n: %d" % (t.n.sum(), t.defaults.sum(), int((t.n == 0).sum()), int(t.n.min())))
    r = []
    for g in sn[D]:
        m1 = d[D] == g; m2 = cc[D] == g
        r.append(dict(debt_ratio_bin=g, n_full=int(m1.sum()), rate_full=float((d.loc[m1, "Default"] == 1).mean()),
                      n_cc=int(m2.sum()), rate_cc=float((cc.loc[m2, "Default"] == 1).mean())))
    r = pd.DataFrame(r); r["diff_cc_minus_full"] = r.rate_cc - r.rate_full
    print("\nDefault rate by Debt Ratio bin, full train vs complete-case subset:"); print(r.to_string(index=False, float_format=lambda x: "%.4f" % x))
    print("overall default rate: full %.4f | complete-case %.4f" % ((d.Default == 1).mean(), (cc.Default == 1).mean()))
    p = ss.check_path(os.path.join(R, "selection/tables/sel_default_cells.csv"))
    data = t.to_csv(index=False).encode()
    with open(p, "xb") as f: f.write(data)
    assert hashlib.sha256(open(p, "rb").read()).hexdigest() == hashlib.sha256(data).hexdigest()
    print("\nsaved sel_default_cells.csv sha12", hashlib.sha256(data).hexdigest()[:12])
_c127()

In [ ]:
# Cell 128 (03_em_score.ipynb): PART E. Run after Cell 127, no restart. Writes sel_bn_test_predictions.pkl and sel_bn_metrics.json (exclusive create).
def _c128():
    import os, sys, json, time, pickle, hashlib, warnings
    warnings.filterwarnings("ignore")
    import numpy as np, pandas as pd
    from sklearn.metrics import roc_auc_score, average_precision_score
    from scipy.stats import ks_2samp
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, os.path.join(R, "shared", "src")); import safe_save as ss
    from pgmpy.models import DiscreteBayesianNetwork
    from pgmpy.inference import VariableElimination, BeliefPropagation
    P_REL, M_REL = "selection/predictions/sel_bn_test_predictions.pkl", "selection/tables/sel_bn_metrics.json"
    for rel in (P_REL, M_REL): ss.check_path(os.path.join(R, rel))
    sn = json.load(open(os.path.join(R, "Data/processed/sel_state_names.json")))
    def conv(df):
        out = {}
        for c in df.columns:
            s = df[c]
            if set(s.dropna().unique()) <= set(sn[c]): out[c] = s
            else:
                m = s.map(lambda x: str(x) if pd.notnull(x) else np.nan)
                assert set(m.dropna().unique()) <= set(sn[c]), "label mismatch in %s: STOP" % c
                out[c] = m
        return pd.DataFrame(out)
    b = pickle.load(open(os.path.join(R, "selection/models/sel_bn_final.pkl"), "rb"))
    model = DiscreteBayesianNetwork([tuple(e) for e in b["model_edges"]]); model.add_cpds(*b["cpds"]); assert model.check_model()
    dte = conv(pd.read_pickle(os.path.join(R, "Data/processed/sel_test.pkl")))
    dtr = conv(pd.read_pickle(os.path.join(R, "Data/processed/sel_train_full.pkl")))
    cols = list(dte.columns); ev_cols = [c for c in cols if c != "Default"]
    A, D = "Age (bin)", "Debt Ratio (bin)"
    cpd = model.get_cpds("Default")
    print("Inference engine for scoring: pgmpy %s VariableElimination, one query per unique evidence pattern (evidence = all observed non-Default variables)" % __import__("pgmpy").__version__)

    def predict(frame):
        codes = np.column_stack([pd.Index(sn[c]).get_indexer(frame[c]) for c in ev_cols])
        u, inv = np.unique(codes, axis=0, return_inverse=True); inv = np.asarray(inv).ravel()
        eng = VariableElimination(model); pu = np.full(len(u), np.nan)
        for j, r in enumerate(u):
            ev = {c: sn[c][k] for c, k in zip(ev_cols, r) if k >= 0}
            try:
                with np.errstate(invalid="ignore", divide="ignore"):
                    p = float(eng.query(variables=["Default"], evidence=ev, show_progress=False).get_value(Default=1))
            except Exception:
                p = np.nan
            pu[j] = p if (np.isfinite(p) and 0 <= p <= 1) else np.nan
        return u, inv, pu, ev_cols
    t0 = time.time(); u, inv, pu, _ = predict(dte)
    print("test: %d unique evidence patterns, VE took %.1fs" % (len(u), time.time() - t0))
    ia, ig = ev_cols.index(A), ev_cols.index(D)
    assert (u[:, ia] >= 0).all() and (u[:, ig] >= 0).all(), "Age/Debt Ratio missing in some test pattern: STOP"
    direct = np.array([float(cpd.get_value(**{"Default": 1, A: sn[A][r[ia]], D: sn[D][r[ig]]})) for r in u])
    valid_u = np.isfinite(pu)
    d1 = float(np.max(np.abs(pu[valid_u] - direct[valid_u])))
    print("VERIFY (i)  max |VE - direct CPD lookup| over %d valid patterns = %.3e" % (valid_u.sum(), d1))
    bad_pat = np.where(~valid_u)[0]
    print("zero-probability (NaN) patterns: %d, rows: %d | direct lookup defined for them: %s" % (
        len(bad_pat), int(np.isin(inv, bad_pat).sum()), bool(np.isfinite(direct[bad_pat]).all()) if len(bad_pat) else "n/a"))
    vidx = np.where(valid_u)[0]
    pick = vidx if len(u) <= 2000 else np.random.RandomState(42).choice(vidx, min(500, len(vidx)), replace=False)
    bp = BeliefPropagation(model); d2, t0 = 0.0, time.time()
    for j in pick:
        ev = {c: sn[c][k] for c, k in zip(ev_cols, u[j]) if k >= 0}
        pb = float(bp.query(variables=["Default"], evidence=ev, show_progress=False).get_value(Default=1))
        d2 = max(d2, abs(pb - pu[j]))
    print("VERIFY (ii) BeliefPropagation (junction tree) on %d patterns (%s): max |BP - VE| = %.3e (%.0fs)" % (
        len(pick), "all" if len(u) <= 2000 else "random 500, seed 42", d2, time.time() - t0))
    if d1 > 1e-10 or d2 > 1e-10: print("STOP: engine mismatch. Not saving."); return None

    p_row = pu[inv]; y_all = dte["Default"].to_numpy().astype(int)
    ok = np.isfinite(p_row); p, y = p_row[ok], y_all[ok]
    print("\nrows excluded for zero-probability evidence: %d of %d" % ((~ok).sum(), len(ok)))
    auc = roc_auc_score(y, p); ks = float(ks_2samp(p[y == 1], p[y == 0]).statistic); ap = average_precision_score(y, p)
    rng = np.random.RandomState(42); n = len(p); bs = np.empty(2000)
    for i in range(2000):
        ix = rng.randint(0, n, n); bs[i] = roc_auc_score(y[ix], p[ix])
    lo, hi = np.percentile(bs, [2.5, 97.5])
    print("AUC %.4f (95%% bootstrap CI %.4f to %.4f) | Gini %.4f | KS %.4f | AP %.4f" % (auc, lo, hi, 2 * auc - 1, ks, ap))
    print("distinct predicted probabilities: %d | mean predicted %.4f vs observed %.4f (n=%d, defaults %d)" % (
        len(np.unique(p)), p.mean(), y.mean(), n, y.sum()))

    TAU = 1 / 6
    utr, invtr, putr, _ = predict(dtr); ptr = putr[invtr]; oktr = np.isfinite(ptr); ptr, ytr = ptr[oktr], dtr["Default"].to_numpy().astype(int)[oktr]
    grid = np.unique(np.concatenate([np.arange(0.005, 1.0, 0.005), [TAU]]))
    cost_tr = np.array([5 * int(((ptr < t) & (ytr == 1)).sum()) + int(((ptr >= t) & (ytr == 0)).sum()) for t in grid])
    best = grid[cost_tr == cost_tr.min()]
    c16 = 5 * int(((ptr < TAU) & (ytr == 1)).sum()) + int(((ptr >= TAU) & (ytr == 0)).sum())
    print("\nTRAIN cross-check (rows used %d of %d): cost-minimising tau on grid = %s (cost %d) | cost at tau=1/6: %d" % (
        oktr.sum(), len(oktr), "%.4f..%.4f" % (best.min(), best.max()) if len(best) > 1 else "%.4f" % best[0], cost_tr.min(), c16))
    pred = p >= TAU
    TP = int((pred & (y == 1)).sum()); FP = int((pred & (y == 0)).sum()); FN = int((~pred & (y == 1)).sum()); TN = int((~pred & (y == 0)).sum())
    cost = 5 * FN + FP
    print("\nTEST at tau_sel = 1/6 = %.6f: TP %d | FP %d | FN %d | TN %d | cost 5*FN+FP = %d" % (TAU, TP, FP, FN, TN, cost))
    print("rows at or above tau: %d (%.2f%% of %d)" % (pred.sum(), 100 * pred.mean(), n))
    print("baselines: accept-all cost %d (5 x %d defaulters) | reject-all cost %d (%d non-defaulters)" % (5 * y.sum(), y.sum(), int((y == 0).sum()), int((y == 0).sum())))
    print("cost saving vs accept-all %.1f%% | vs reject-all %.1f%%" % (100 * (1 - cost / (5 * y.sum())), 100 * (1 - cost / (y == 0).sum())))
    print("\ncells (Age x Debt Ratio) with P(Default=1) >= 1/6:")
    above = []
    for a in sn[A]:
        for g in sn[D]:
            v = float(cpd.get_value(**{"Default": 1, A: a, D: g}))
            if v >= TAU:
                nte = int(((dte[A] == a) & (dte[D] == g)).sum()); above.append((a, g, round(v, 4), nte))
    for c_ in above: print("   ", c_)
    if not above: print("    none")

    pk = ss.check_path(os.path.join(R, P_REL))
    blob = pickle.dumps(dict(index=list(dte.index), y=y_all, p=p_row, included=ok, tau=TAU, engine="pgmpy VariableElimination"))
    with open(pk, "xb") as f: f.write(blob)
    met = dict(engine="pgmpy VariableElimination", n_test=len(ok), n_excluded=int((~ok).sum()), auc=auc, auc_ci95=[float(lo), float(hi)],
               gini=2 * auc - 1, ks=ks, ap=ap, distinct_probs=int(len(np.unique(p))), mean_pred=float(p.mean()), obs_rate=float(y.mean()),
               verify_direct_maxdiff=d1, verify_bp_maxdiff=d2, bp_patterns=int(len(pick)), tau_sel=TAU,
               train_best_tau=[float(best.min()), float(best.max())], train_cost_min=int(cost_tr.min()), train_cost_at_tau=c16,
               test=dict(TP=TP, FP=FP, FN=FN, TN=TN, cost=int(cost), n_at_or_above=int(pred.sum()), share_at_or_above=float(pred.mean()),
                         accept_all_cost=int(5 * y.sum()), reject_all_cost=int((y == 0).sum())),
               cells_at_or_above_tau=[list(map(str, c_)) for c_ in above])
    mp = ss.check_path(os.path.join(R, M_REL)); mb = json.dumps(met, indent=1).encode()
    with open(mp, "xb") as f: f.write(mb)
    print("\nsaved sel_bn_test_predictions.pkl (%s), sel_bn_metrics.json (%s)" % (hashlib.sha256(blob).hexdigest()[:12], hashlib.sha256(mb).hexdigest()[:12]))
    print("PART E DONE")
_c128()

In [ ]:
# Cell 129 (03_em_score.ipynb): save sel_blanket_robustness.csv (Part A). Run after Cell 128, no restart.
def _c129():
    import os, sys, json, hashlib, warnings
    warnings.filterwarnings("ignore")
    import pandas as pd
    R = os.environ["CREDIT_BN_ROOT"]
    assert os.environ.get("PYTHONHASHSEED") == "0", "wrong kernel: STOP"
    sys.path.insert(0, os.path.join(R, "shared", "src"))
    import structure, safe_save as ss
    out = ss.check_path(os.path.join(R, "selection/tables/sel_blanket_robustness.csv"))
    order = json.load(open(os.path.join(R, "Data/processed/sel_node_order.json")))
    nodes7 = list(order["derived_7_nodes_selection"])
    data = pd.read_pickle(os.path.join(R, "Data/processed/sel_train_cc.pkl"))[nodes7]
    assert data.shape == (96251, 7)
    TARGET = {"Age (bin)", "Debt Ratio (bin)"}
    st = pd.read_csv(os.path.join(R, "selection/tables/sel_structure_seed_table.csv")).set_index("seed")
    ids, rows = {}, []
    for s in range(42, 57):
        r = structure.learn_structure(data, nodes7, seeds=[s], edge_density=0.15, verbose=False)
        E = sorted((str(u), str(v)) for u, v in r["best_dag"].edges())
        assert abs(r["best_score"] - float(st.loc[s, "BIC"])) < 1e-3, "BIC mismatch seed %d: STOP" % s
        sid = ids.setdefault(hashlib.md5(str(E).encode()).hexdigest()[:8], len(ids) + 1)
        par = sorted(u for u, v in E if v == "Default"); chi = sorted(v for u, v in E if u == "Default")
        cop = sorted({u for u, v in E if v in chi and u != "Default"})
        bl = sorted(set(par) | set(chi) | set(cop))
        rows.append(dict(seed=s, bic=round(r["best_score"], 4), n_edges=len(E), structure_id=sid,
                         default_parents="|".join(par), default_children="|".join(chi), default_coparents="|".join(cop),
                         blanket="|".join(bl), blanket_equals_age_debt_ratio=set(bl) == TARGET,
                         is_chosen=(s == 55), is_runner_up=(s == 50)))
    df = pd.DataFrame(rows)
    t = df[df.blanket_equals_age_debt_ratio]
    print("seeds with blanket {Age, Debt Ratio}: %d / 15 | distinct structures with it: %d | overall distinct: %d" % (
        len(t), t.structure_id.nunique(), df.structure_id.nunique()))
    print("seeds:", t.seed.tolist(), "| runner-up (seed 50) has it:", bool(df.loc[df.seed == 50, "blanket_equals_age_debt_ratio"].iloc[0]))
    data_b = df.to_csv(index=False).encode()
    with open(out, "xb") as f: f.write(data_b)
    assert hashlib.sha256(open(out, "rb").read()).hexdigest() == hashlib.sha256(data_b).hexdigest()
    print("saved sel_blanket_robustness.csv sha12", hashlib.sha256(data_b).hexdigest()[:12])
_c129()

In [ ]:
# Cell 130 (03_em_score.ipynb): Phase 4 manifest/git DRY RUN. Writes NOTHING. No restart needed.
def _c130():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    TAG12 = "5626192403ef"
    def git(*a):
        r = subprocess.run(["git", "-C", R, *a], capture_output=True, text=True)
        return r.stdout.strip()
    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 20), b''):
                h.update(b)
        return h.hexdigest()
    mp = os.path.join(R, "MANIFEST.csv")
    raw = open(mp, 'rb').read()
    existing = {r['path'] for r in csv.DictReader(open(mp, newline=''))}
    print("DRY_RUN = True")
    print("manifest rows now:", len(existing), "| ends with newline:", raw.endswith(b'\n'))
    print("tag points to:", git("rev-list", "-n", "1", "full-variable-final")[:12], "| OK:", git("rev-list", "-n", "1", "full-variable-final").startswith(TAG12))
    print("git status before:", git("status", "--porcelain").splitlines())

    fixed = ["shared/src/loglik.py", "shared/src/em.py",
             "selection/notebooks/03_em_score.ipynb",
             "selection/tables/sel_em_history.csv", "selection/tables/sel_em_zero_weight_rows.csv",
             "selection/tables/sel_default_cells.csv", "selection/tables/sel_bn_metrics.json",
             "selection/tables/sel_blanket_robustness.csv",
             "selection/figures/sel_em_convergence.png",
             "selection/predictions/sel_bn_test_predictions.pkl",
             "selection/models/sel_bn_final.pkl", "selection/models/sel_bn_final.json"]
    models = sorted("selection/models/" + f for f in os.listdir(os.path.join(R, "selection/models")) if f.startswith("sel_em_ckpt"))
    want = fixed + models
    rows = []
    for rel in want:
        p = os.path.join(R, rel)
        if not os.path.exists(p):
            print("MISSING", rel); continue
        if rel in existing:
            print("ALREADY IN MANIFEST", rel); continue
        rows.append((rel, os.path.getsize(p), sha(p)[:12], time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p)))))
    print("\nrows to append (%d):" % len(rows))
    for r in rows: print("  ", *r)
    print("expected manifest rows after:", len(existing) + len(rows))

    # anything new on disk under selection/ or shared/ that is NOT in the manifest or the list above?
    stray = []
    for top in ("selection", "shared"):
        for dp, dn, fn in os.walk(os.path.join(R, top)):
            dn[:] = [d for d in dn if d not in (".ipynb_checkpoints", "__pycache__")]
            for f in fn:
                rel = os.path.relpath(os.path.join(dp, f), R)
                if rel not in existing and rel not in want and not f.endswith(".pyc"):
                    stray.append(rel)
    print("files on disk not in manifest and not in the list:", stray if stray else "none")
    print("\nwould commit: MANIFEST.csv, shared/src/loglik.py, shared/src/em.py, selection/notebooks/03_em_score.ipynb (data/pkl/csv/png/json are git-ignored by design)")
    print("DRY RUN ONLY: nothing written.")
_c130()

In [ ]:
# Cell 131 v2 (03_em_score.ipynb): REAL RUN (approved). Same as Cell 131, but the notebook's hash is recorded as-is (Jupyter autosave changes it).
def _c131():
    import os, csv, time, hashlib, subprocess
    R = os.environ["CREDIT_BN_ROOT"]
    TAG12 = "5626192403ef"
    NOTE = "added after Phase 4, 2026-10-10"
    NB = "selection/notebooks/03_em_score.ipynb"
    FILES = [("shared/src/loglik.py", "shared", "cdc26c96a8d6"),
             ("shared/src/em.py", "shared", "c245161bebeb"),
             (NB, "selection", None),
             ("selection/tables/sel_em_history.csv", "selection", "eea28204292d"),
             ("selection/tables/sel_em_zero_weight_rows.csv", "selection", "c857b65fa097"),
             ("selection/tables/sel_default_cells.csv", "selection", "098850dce6ba"),
             ("selection/tables/sel_bn_metrics.json", "selection", "06e6aee0ede1"),
             ("selection/tables/sel_blanket_robustness.csv", "selection", "6cdbb45e6188"),
             ("selection/figures/sel_em_convergence.png", "selection", "d8e0195f9adc"),
             ("selection/predictions/sel_bn_test_predictions.pkl", "selection", "c27a6366bd5a"),
             ("selection/models/sel_bn_final.pkl", "selection", "804a1ffc4309"),
             ("selection/models/sel_bn_final.json", "selection", "361fd4c23375"),
             ("selection/models/sel_em_ckpt_final.pkl", "selection", "d0e6ff1a2296"),
             ("selection/models/sel_em_ckpt_it005.pkl", "selection", "1a9e20632524")]
    def git(*a, check=True):
        r = subprocess.run(["git", "-C", R, *a], capture_output=True, text=True)
        if check and r.returncode != 0:
            raise RuntimeError(f"git {' '.join(a)} failed: {r.stderr.strip()[:300]}")
        return r.stdout.strip()
    def sha(p):
        h = hashlib.sha256()
        with open(p, 'rb') as f:
            for b in iter(lambda: f.read(1 << 20), b''):
                h.update(b)
        return h.hexdigest()

    mp = os.path.join(R, "MANIFEST.csv")
    raw0 = open(mp, 'rb').read()
    existing = {r['path'] for r in csv.DictReader(open(mp, newline=''))}
    assert raw0.endswith(b'\n') and len(existing) == 134, "unexpected manifest state: STOP"
    assert git("rev-list", "-n", "1", "full-variable-final").startswith(TAG12), "tag moved: STOP"

    rows = []
    for rel, role, h12 in FILES:
        p = os.path.join(R, rel)
        assert rel not in existing and os.path.exists(p), f"unexpected state for {rel}: STOP"
        h = sha(p)
        if h12 is not None:
            assert h.startswith(h12), f"hash changed since dry run for {rel}: STOP"
        else:
            print("notebook hash recorded now: %s (dry run showed 836cad2f52a7)" % h[:12])
        rows.append([rel, os.path.getsize(p), h, role, time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(p))), NOTE])
    with open(mp, 'a', newline='') as fh:
        csv.writer(fh).writerows(rows)
    n = len(list(csv.DictReader(open(mp, newline=''))))
    assert n == 148 and open(mp, 'rb').read().startswith(raw0), "manifest check failed: STOP (not committed)"

    git("add", "MANIFEST.csv", "shared/src/loglik.py", "shared/src/em.py", NB)
    ident = []
    if not git("config", "user.name", check=False) or not git("config", "user.email", check=False):
        ident = ["-c", "user.name=Ahmed Nanabhay", "-c", "user.email=noreply@local.invalid"]
    git(*ident, "commit", "-q", "-m", "Phase 4: loglik.py, em.py, EM fit, scoring outputs added to manifest")
    print("manifest rows now:", n, "(expected 148)")
    print("new commit:", git("log", "--oneline", "-1"))
    print("tag unchanged:", git("rev-list", "-n", "1", "full-variable-final").startswith(TAG12))
    print("git status:", git("status", "--porcelain").splitlines())
    print("notebook hash after commit: %s (a difference from the recorded one just means Jupyter autosaved again)" % sha(os.path.join(R, NB))[:12])

_c131()